# Flood summaries for ThinkHazard

This script performs flood hazard ranking by administrative unit using global-extent
Fathom tiles hosted on AWS S3, rather than country-extent locally downloaded data.

The hazard ranking is based on:
- Value threshold: Minimum flood depth (cm) to consider (50 cm)
- Area threshold: Minimum percentage of area affected (3%)
- Hazard score: Count of return periods meeting BOTH thresholds (0-4)

**CORRECTED VERSION**: Fixed windowing bugs and added proper hazard score calculation

In [ ]:
import os, time, io, json, sys
import urllib3
import boto3

import geopandas as gpd
import pandas as pd
import numpy as np

from functools import reduce
from urllib3.exceptions import InsecureRequestWarning
from botocore import UNSIGNED
from botocore.config import Config
from tqdm.notebook import tqdm

# Import helper functions
from gfdrr_helper import *

urllib3.disable_warnings(InsecureRequestWarning)

def tPrint(s):
    """prints the time along with the message"""
    print("%s\t%s" % (time.strftime("%H:%M:%S"), s))

s3_client = boto3.client('s3', verify=False, config=Config(signature_version=UNSIGNED))

%load_ext autoreload
%autoreload 2

In [ ]:
# Configuration
local_folder = "C:/WBG/Work/Projects/ThinkHazard"
out_folder = os.path.join(local_folder, "FATHOM_summaries")
map_folder = os.path.join(local_folder, "FATHOM_maps")
for tF in [out_folder, map_folder]:
    if not os.path.exists(tF):
        os.makedirs(tF)

vrt_folder = r"C:\WBG\Work\data\FATHOM"
s3_bucket = "wbg-geography01"
s3_prefix = "FATHOM"

# Return periods to process
return_periods = [10, 100, 500, 1000]

# Flood types and their VRT file patterns
flood_files = [
    ["FU", "FLOOD_MAP-1ARCSEC-NW_OFFSET-1in{rp}-FLUVIAL-UNDEFENDED-DEPTH-2020-PERCENTILE50-v3.1.vrt"],
    ["CU", "FLOOD_MAP-1ARCSEC-NW_OFFSET-1in{rp}-COASTAL-UNDEFENDED-DEPTH-2020-PERCENTILE50-v3.1.vrt"],
    ['PD', "FLOOD_MAP-1ARCSEC-NW_OFFSET-1in{rp}-PLUVIAL-DEFENDED-DEPTH-2020-PERCENTILE50-v3.1.vrt"]
]

# Load administrative boundaries
admin_boundaries_file = r"C:\WBG\Work\data\ADMIN\NEW_WB_BOUNDS\FOR_PUBLICATION\crs_4326\parquet\WB_GAD_ADM2.parquet"
inA = gpd.read_parquet(admin_boundaries_file)

print(f"Loaded {len(inA)} administrative units")
print(f"Countries: {len(inA['ISO_A3'].unique())}")

In [ ]:
# Main processing cell - CORRECTED VERSION

cur_out_folder = os.path.join(out_folder, "FATHOM_Detailed")
if not os.path.exists(cur_out_folder):
    os.makedirs(cur_out_folder)

cur_map_folder = os.path.join(map_folder, "FATHOM_Detailed")
if not os.path.exists(cur_map_folder):
    os.makedirs(cur_map_folder)

# Define thresholds for hazard scoring
VALUE_THRESHOLD = 50   # cm - minimum mean depth
AREA_THRESHOLD = 3.0   # % - minimum area fraction

tPrint(f"Thresholds: Value={VALUE_THRESHOLD} cm, Area={AREA_THRESHOLD}%")

with rasterio.Env(GDAL_HTTP_UNSAFESSL='YES'):
    for sel_country in ['TUN','GEO','PHL']: #inA['ISO_A3'].unique():
        all_res = []
        out_file = os.path.join(cur_out_folder, f"FATHOM_ThinkHazard_summary_{sel_country}.csv")
        sel_a = inA[inA['ISO_A3'] == sel_country]
        
        if not os.path.exists(out_file) and not (sel_country in ["FJI",'RUS']):
            tPrint(f"Processing country: {sel_country} ({len(sel_a)} units)")
            
            # Process each flood type and return period
            for lbl, raster_file in flood_files:
                for return_period in return_periods:
                    tPrint(f"Processing {lbl} for {return_period} year return period")
                    sel_raster_file = raster_file.format(rp=return_period)
                    sel_raster = f"s3://{s3_bucket}/{s3_prefix}/{sel_raster_file}"
                    
                    # CORRECTED: Removed no_data parameter (now handled automatically with data < 0)
                    res_a = calculate_think_hazard_score(sel_a, sel_raster,
                                                         depth_threshold=50,
                                                         idx_col='ADM2CD_c',
                                                         all_touched=True)
                    
                    res_a.rename(columns={
                        'frac_area_flooded': f'frac_area_flooded_{lbl}_{return_period}yr',
                        'mean_val': f'mean_val_{lbl}_{return_period}yr'
                    }, inplace=True)
                    all_res.append(res_a)
            
            # Merge all results
            tPrint("Merging results...")
            all_res_df = reduce(lambda left, right: pd.merge(left, right, on='ADM2CD_c', how='outer'), all_res)
            
            # ADDED: Calculate hazard scores based on dual thresholds
            tPrint("Calculating hazard scores...")
            for lbl, _ in flood_files:
                # Get columns for this flood type across all return periods
                rp_cols = [(f'mean_val_{lbl}_{rp}yr', f'frac_area_flooded_{lbl}_{rp}yr')
                           for rp in return_periods]
                
                # Calculate score: count return periods where BOTH thresholds are exceeded
                all_res_df[f'{lbl}_score'] = all_res_df.apply(
                    lambda row: sum(
                        1 for mean_col, area_col in rp_cols
                        if (row.get(mean_col, 0) >= VALUE_THRESHOLD and 
                            row.get(area_col, 0) >= AREA_THRESHOLD)
                    ),
                    axis=1
                )
                
                tPrint(f"  {lbl} scores: {all_res_df[f'{lbl}_score'].value_counts().sort_index().to_dict()}")
            
            # Fill NaN values with 0
            all_res_df.fillna(0, inplace=True)
            
            # Save results
            all_res_df.to_csv(out_file, index=False)
            tPrint(f"Saved results to {out_file}")
            
            # Create map
            sel_a = inA[inA['ISO_A3'] == sel_country]
            map_adm = pd.merge(sel_a, all_res_df, on='ADM2CD_c', how='left')
            map_file = os.path.join(cur_map_folder, f"flood_map_{sel_country}_100yr.png")
            map_flood(map_adm, return_period=100, out_file=map_file)
            tPrint(f"Saved map to {map_file}")
        else:
            if os.path.exists(out_file):
                tPrint(f"File already exists for {sel_country}, skipping...")
            else:
                tPrint(f"Skipping {sel_country} (in exclusion list)")

tPrint("Processing complete!")

## Verification: Check results for a specific country

In [ ]:
# Load and inspect results
sel_country = 'TUN'
result_file = os.path.join(cur_out_folder, f"FATHOM_ThinkHazard_summary_{sel_country}.csv")

if os.path.exists(result_file):
    results = pd.read_csv(result_file)
    print(f"Results for {sel_country}: {len(results)} units")
    print("\nColumns:", list(results.columns))
    print("\nSample results:")
    display(results.head())
    
    print("\nHazard Score Distribution:")
    for flood_type in ['FU', 'CU', 'PD']:
        score_col = f'{flood_type}_score'
        if score_col in results.columns:
            print(f"\n{flood_type}:")
            print(results[score_col].value_counts().sort_index())
else:
    print(f"Results file not found: {result_file}")

## Verification: Check specific units with known values

Expected values for Tunisia Coastal RP1000:
- TUN016015: area ~4.5% (not 1.57%)
- TUN016001: area ~2.9% (not 0.23%)

In [ ]:
# Check specific units
test_units = ['TUN016015', 'TUN016001']

if os.path.exists(result_file):
    results = pd.read_csv(result_file)
    test_results = results[results['ADM2CD_c'].isin(test_units)]
    
    if len(test_results) > 0:
        print("Test units - Coastal RP1000:")
        for _, row in test_results.iterrows():
            unit = row['ADM2CD_c']
            mean_val = row.get('mean_val_CU_1000yr', 0)
            area_pct = row.get('frac_area_flooded_CU_1000yr', 0)
            score = row.get('CU_score', 0)
            print(f"  {unit}: mean={mean_val:.2f} cm, area={area_pct:.2f}%, score={score}")
        
        print("\nExpected:")
        print("  TUN016015: area ~4.5%")
        print("  TUN016001: area ~2.9%")
    else:
        print("Test units not found in results")
else:
    print(f"Results file not found")

## Optional: Extract sample data for testing

In [ ]:
# Uncomment and run if you need to extract sample raster data

# sys.path.insert(0, "C:\\WBG\\Work\\Code\\GOSTrocks\\src")
# import GOSTrocks.rasterMisc as rMisc

# temp_out_folder = "C:/Temp/FATHOM_TUN"
# if not os.path.exists(temp_out_folder):
#     os.makedirs(temp_out_folder)

# sel_admin = inA.loc[inA['ISO_A3'] == "TUN"]
# sel_admin.to_file(os.path.join(temp_out_folder, "TUN_admin.gpkg"), driver="GPKG")

# for return_period in return_periods:
#     for lbl, raster_file in flood_files:
#         temp_out_file = os.path.join(temp_out_folder, f"TUN_{lbl}_{return_period}yr.tif")
#         if not os.path.exists(temp_out_file):
#             sel_raster_file = raster_file.format(rp=return_period)
#             sel_raster = f"s3://{s3_bucket}/{s3_prefix}/{sel_raster_file}"
#             with rasterio.Env(GDAL_HTTP_UNSAFESSL='YES'):
#                 inR = rasterio.open(sel_raster)
#                 rMisc.clipRaster(inR, sel_admin, temp_out_file, crop=False)